In [1]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Stepang08/Plant_Disease_Detection.git 2>/dev/null; true
%cd /content/Plant_Disease_Detection
!git pull

!pip install -q timm scikit-learn pyyaml wandb

!mkdir -p data/raw data/processed /content/dataset models
!unzip -qo "/content/drive/.shortcut-targets-by-id/1gaVEvJyVG2sQNDgJ_0gSyGHx3s2S5DjQ/PlantDoc/train.zip" -d /content/dataset/
!unzip -qo "/content/drive/.shortcut-targets-by-id/1gaVEvJyVG2sQNDgJ_0gSyGHx3s2S5DjQ/PlantDoc/val.zip" -d /content/dataset/
!ln -sf /content/dataset/train data/raw/train
!ln -sf /content/dataset/val data/raw/val
!python -m src.dataset


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/Plant_Disease_Detection
Already up to date.


In [2]:
import json, time, numpy as np, torch, torch.nn as nn
import wandb
from torch.utils.data import DataLoader
from src.dataset import PlantDiseaseDataset
from src.transforms import build_val_transforms
from src.utils import compute_mAP, set_seed

set_seed(42)
device = torch.device("cuda")

# W&B init
wandb.init(project="plant-disease-detection", config={
    "model": "dinov2_vitb14",
    "trainable_params": "30K",
    "method": "frozen_backbone_linear_head",
    "lr": 1e-3,
    "epochs": 50,
    "label_smoothing": 0.1,
})

# Load DINOv2
print("Loading DINOv2...")
backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
backbone = backbone.to(device).eval()
embed_dim = backbone.embed_dim

# Extract features
transform = build_val_transforms(224)
train_ds = PlantDiseaseDataset(split="train", transform=transform)
val_ds = PlantDiseaseDataset(split="val", transform=transform)

def extract_features(dataset, backbone, device, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    all_feats, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            feats = backbone(imgs.to(device))
            all_feats.append(feats.cpu())
            all_labels.append(labels)
    return torch.cat(all_feats), torch.cat(all_labels)

print("Extracting features...")
train_feats, train_labels = extract_features(train_ds, backbone, device)
val_feats, val_labels = extract_features(val_ds, backbone, device)

# Train linear head with W&B logging
num_classes = 39
head = nn.Linear(embed_dim, num_classes).to(device)
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

train_feats_gpu = train_feats.to(device)
train_labels_gpu = train_labels.to(device)

print("Training...")
best_mAP = 0
for epoch in range(50):
    head.train()
    logits = head(train_feats_gpu)
    loss = criterion(logits, train_labels_gpu)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Validate every epoch
    head.eval()
    with torch.no_grad():
        val_logits = head(val_feats.to(device))
        val_probs = torch.softmax(val_logits, dim=1).cpu().numpy()
    metrics = compute_mAP(val_labels.numpy(), val_probs, num_classes)

    wandb.log({
        "epoch": epoch,
        "train_loss": loss.item(),
        "val_mAP": metrics["mAP"],
        "val_top1_acc": metrics["top1_acc"],
    })

    if metrics["mAP"] > best_mAP:
        best_mAP = metrics["mAP"]

    if epoch % 10 == 0 or epoch == 49:
        print(f"  epoch {epoch:3d} | loss {loss.item():.4f} | val_mAP {metrics['mAP']:.4f} | top1 {metrics['top1_acc']:.4f}")

print(f"\n=== FINAL: mAP={best_mAP:.4f} ===")

# Save model
from src.model import DINOv2Model
full_model = DINOv2Model(num_classes=39, dropout=0.0).to(device)
full_model.head[1].weight.data = head.weight.data.clone()
full_model.head[1].bias.data = head.bias.data.clone()
model_state = {
    "epoch": 49,
    "model": full_model.state_dict(),
    "best_mAP": best_mAP,
    "config": {
        "model": {"backbone": "dinov2_vitb14", "num_classes": 39, "pretrained": True, "dropout": 0.0},
        "data": {"image_size": 224},
    },
}
torch.save(model_state, "models/best_model.pth")
!cp models/best_model.pth /content/drive/MyDrive/plant_disease_data/checkpoints/dinov2_best.pth

wandb.log({"best_mAP": best_mAP})
wandb.finish()
print("Done! Model saved, W&B run finished.")


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: stepan-goyunyan (stepan-goyunyan-physmath-school-after-a-shahinyan-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loading DINOv2...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Extracting features...
Training...
  epoch   0 | loss 4.1417 | val_mAP 0.0686 | top1 0.0848
  epoch  10 | loss 1.7539 | val_mAP 0.6406 | top1 0.6207
  epoch  20 | loss 1.4567 | val_mAP 0.7695 | top1 0.7308
  epoch  30 | loss 1.3009 | val_mAP 0.8247 | top1 0.7700
  epoch  40 | loss 1.1936 | val_mAP 0.8485 | top1 0.7879
  epoch  49 | loss 1.1353 | val_mAP 0.8625 | top1 0.8051

=== FINAL: mAP=0.8625 ===


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


best_mAP,▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇███
train_loss,█▇▆▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_mAP,▁▂▂▃▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇███████████████████
val_top1_acc,▁▂▃▃▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇███████████████████
best_mAP,0.86252
epoch,49
train_loss,1.13529
val_mAP,0.86252
val_top1_acc,0.80506


Done! Model saved, W&B run finished.


In [3]:
!python -m src.evaluate --checkpoint models/best_model.pth
!echo "=== WITH TTA ==="
!python -m src.evaluate --checkpoint models/best_model.pth --tta


Device: cuda
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Evaluating on val: 1226 images

=== Results (val, TTA=False) ===
mAP:       0.8625
Top-1 acc: 0.8051
Top-5 acc: 0.9861

--- Weakest 10 classes ---
  0.3614  bacterial leaf spot
  0.4368  bacterial leaf streak (black chaff)
  0.6287  septoria leaf spot
  0.6570  septoria blotch
  0.7059  brown spot
  0.7505  mosaic
  0.7668  angular leaf spot
  0.7886  leaf